# Base imports

In [1]:
library(devtools)

Loading required package: usethis



In [2]:
library(tidyverse)

── Attaching core tidyverse packages ───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.2     ✔ tibble    3.2.1
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.0.4     
── Conflicts ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


In [3]:
library(here)

here() starts at /n/scratch/users/r/rob6090/projects/tdp-43/scripts/analysis/final_dsRNA



In [4]:
library(powerjoin)

In [5]:
library(qs)

qs 0.27.3. Announcement: https://github.com/qsbase/qs/issues/103



In [6]:
library(Rsubread)

In [7]:
library(ggbeeswarm)

In [8]:
library(Rsubread)

In [9]:
library(GenomicRanges)

Loading required package: stats4

Loading required package: BiocGenerics


Attaching package: ‘BiocGenerics’


The following objects are masked from ‘package:lubridate’:

    intersect, setdiff, union


The following objects are masked from ‘package:dplyr’:

    combine, intersect, setdiff, union


The following objects are masked from ‘package:stats’:

    IQR, mad, sd, var, xtabs


The following objects are masked from ‘package:base’:

    anyDuplicated, aperm, append, as.data.frame, basename, cbind,
    colnames, dirname, do.call, duplicated, eval, evalq, Filter, Find,
    get, grep, grepl, intersect, is.unsorted, lapply, Map, mapply,
    match, mget, order, paste, pmax, pmax.int, pmin, pmin.int,
    Position, rank, rbind, Reduce, rownames, sapply, setdiff, sort,
    table, tapply, union, unique, unsplit, which.max, which.min


Loading required package: S4Vectors


Attaching package: ‘S4Vectors’


The following objects are masked from ‘package:lubridate’:

    second, second<-


The

In [12]:
# Prepare BAM files
bam_files <- tibble(
  bam_path = Sys.glob(file.path("/home/rob6090/scr_proj/tdp-43/data/samples_bam/*.sorted.bam"))
) %>%
  filter(!str_detect(bam_path, fixed("markdup"))) %>%
  mutate(
    sample_id = str_extract(bam_path, "([^/]+)\\.primary.sorted\\.bam", group = 1)
  )

In [13]:
bam_files

bam_path,sample_id
<chr>,<chr>
/home/rob6090/scr_proj/tdp-43/data/samples_bam/SRR8571937.sorted.bam,NA
/home/rob6090/scr_proj/tdp-43/data/samples_bam/SRR8571938.sorted.bam,NA
/home/rob6090/scr_proj/tdp-43/data/samples_bam/SRR8571939.sorted.bam,NA
/home/rob6090/scr_proj/tdp-43/data/samples_bam/SRR8571940.sorted.bam,NA
/home/rob6090/scr_proj/tdp-43/data/samples_bam/SRR8571941.sorted.bam,NA
/home/rob6090/scr_proj/tdp-43/data/samples_bam/SRR8571942.sorted.bam,NA
/home/rob6090/scr_proj/tdp-43/data/samples_bam/SRR8571944.sorted.bam,NA
/home/rob6090/scr_proj/tdp-43/data/samples_bam/SRR8571945.sorted.bam,NA
/home/rob6090/scr_proj/tdp-43/data/samples_bam/SRR8571947.sorted.bam,NA


In [14]:
# Load RepeatMasker file
repeatmasker_raw <- read_csv("~/main/projects/genomics/tdp-43/data/repeatmasker_raw.csv") # Assume file named repeatmasker_raw.csv

Rows: 5448004 Columns: 16
── Column specification ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: ","
chr (8): chromosome, left, strand, rep_name, class_family, rep_start, rep_le...
dbl (8): sw_score, perc_div, perc_del, perc_ins, start, end, rep_end, id

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [15]:
repeats_saf <- repeatmasker_raw%>%
  transmute(
    GeneID = paste(rep_name, row_number(), sep = "_"),
    Chr = chromosome,
    Start = start,
    End = end,
    Strand = case_when(
      strand == "+" ~ "+",
      strand == "C" ~ "-",
      TRUE ~ "*"
    )
  )

In [16]:
# Run featureCounts
fc_results <- featureCounts(
  files = bam_files$bam_path,
  annot.ext = repeats_saf,
  isGTFAnnotationFile = FALSE,
  allowMultiOverlap = TRUE,
  primaryOnly = TRUE,
  isPairedEnd = TRUE,
  nthreads = 14
)


        ==========     _____ _    _ ____  _____  ______          _____  
        =====         / ____| |  | |  _ \|  __ \|  ____|   /\   |  __ \ 
          =====      | (___ | |  | | |_) | |__) | |__     /  \  | |  | |
            ====      \___ \| |  | |  _ <|  _  /|  __|   / /\ \ | |  | |
              ====    ____) | |__| | |_) | | \ \| |____ / ____ \| |__| |
        ==========   |_____/ \____/|____/|_|  \_\______/_/    \_\_____/
       Rsubread 2.16.1

//========================== featureCounts setting ===========================\\
||                                                                            ||
||             Input files : 14 BAM files                                     ||
||                                                                            ||
||                           SRR8571937.sorted.bam                            ||
||                           SRR8571938.sorted.bam                            ||
||                           SRR8571939.sorted.bam  

In [17]:
qsave(
  fc_results,
  "/n/scratch/users/r/rob6090/projects/tdp-43/scripts/analysis/final_dsRNA/liu_repeat_counts_raw-primary_only_mod.qs"
)